<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/06_text_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chexpert_plus_manifest.parquet"
)

manifest = pd.read_parquet(MANIFEST_PATH)
print("Manifest loaded successfully.")
print("Shape:", manifest.shape)

Mounted at /content/drive
Manifest loaded successfully.
Shape: (223462, 54)


In [2]:
print("Report dtype:", manifest["report"].dtype)
print("Missing reports:", manifest["report"].isna().sum())
print("Empty reports:", (manifest["report"].str.strip() == "").sum())

for i, report in enumerate(manifest["report"].head(5)):
    print(f"\n--- Report {i + 1} ---")
    print(report)

Report dtype: object
Missing reports: 0
Empty reports: 0

--- Report 1 ---
NARRATIVE:
CHEST, ONE VIEW: 2-10-2001
FINDINGS: Costophrenic angles sharp, without evidence of effusion.
The cardiomediastinal silhouette is normal. Vessels mildly
indistinct with prominence of interstitial structures, suggesting
mild, pulmonary edema. Left subclavian central venous catheter is
seen, tip in mid SVC. No pneumothorax.
IMPRESSION:
1. NO EVIDENCE OF PNEUMOTHORAX.
2. MILD INTERSTITIAL PULMONARY EDEMA.
END OF IMPRESSION:
SUMMARY: 2
I have personally reviewed the images for this examination and agree
with the report transcribed above.
By: Dr. Juarez Tucker N  on: 2-10-2001
 __________________________________
 
ACCESSION NUMBER:
LFEWOZWVDRX
This report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.

--- Report 2 ---
NARRATIVE:
CHEST X-RAY: 2/21/2006
COMPARISON: 2/21/2006.
CLINICAL HISTORY: Dyspnea. Multiple myeloma.
IMPRESSION:
1. PA AND 

In [3]:
manifest["report_char_length"] = manifest["report"].str.len()

print("--- Character Length Describe ---")
print(manifest["report_char_length"].describe())

print("\n--- Character Length Percentiles ---")
print(
    manifest["report_char_length"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 1.00]
    )
)

--- Character Length Describe ---
count    223462.000000
mean        842.414858
std         271.541716
min         254.000000
25%         673.000000
50%         789.000000
75%         943.000000
max        5662.000000
Name: report_char_length, dtype: float64

--- Character Length Percentiles ---
0.500     789.000
0.750     943.000
0.900    1162.000
0.950    1347.000
0.990    1818.000
0.995    2043.695
1.000    5662.000
Name: report_char_length, dtype: float64


In [4]:
manifest["report_word_length"] = manifest["report"].str.split().str.len()

print("--- Word Length Describe ---")
print(manifest["report_word_length"].describe())

print("\n--- Word Length Percentiles ---")
print(
    manifest["report_word_length"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 1.00]
    )
)

--- Word Length Describe ---
count    223462.000000
mean        117.365315
std          39.862881
min          36.000000
25%          92.000000
50%         110.000000
75%         132.000000
max         828.000000
Name: report_word_length, dtype: float64

--- Word Length Percentiles ---
0.500    110.0
0.750    132.0
0.900    164.0
0.950    191.0
0.990    261.0
0.995    294.0
1.000    828.0
Name: report_word_length, dtype: float64


In [5]:
print("--- Shortest Reports ---")
print(
    manifest[
        ["report", "report_char_length", "report_word_length"]
    ]
    .sort_values("report_char_length")
    .head(10)
    .to_string(index=False)
)

print("\n--- Longest Reports ---")
print(
    manifest[
        ["report", "report_char_length", "report_word_length"]
    ]
    .sort_values("report_char_length", ascending=False)
    .head(10)
    .to_string(index=False)
)

--- Shortest Reports ---
                                                                                                                                                                                                                                                                                                              report  report_char_length  report_word_length
                                              NARRATIVE:\nIMPRESSION:\nINTERSTITIAL EDEMA.\nEND OF IMPRESSION\nSUMMARY: 2 ABNORMAL, PREVIOUSLY REPORTED\n \nACCESSION NUMBER:\nTMKPNPH\nThis report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.                 254                  36
                        NARRATIVE:\nEXAMINATION:\nCHEST PA AND LATERAL\nCOMPARISON: NONE\nIMPRESSION:\nNORMAL CHEST. NO EVIDENCE OF PNEUMONIA.\nSUMMARY: 1\n \nACCESSION NUMBER:\noose\nThis report has been anonymized. All dates are offset from the actual dates by a fixed interv

In [6]:

duplicate_reports = manifest["report"].duplicated().sum()
unique_reports = manifest["report"].nunique()

print(f"Unique reports: {unique_reports:,}")
print(f"Duplicate report rows: {duplicate_reports:,}")

print("\n--- Top 10 Most Frequent Reports ---")
print(manifest["report"].value_counts().head(10))

Unique reports: 223,460
Duplicate report rows: 2

--- Top 10 Most Frequent Reports ---
report
NARRATIVE:\nCOMPARISON: None available.\n \nIMPRESSION:\n \nCARDIOMEDIASTINAL SILHOUETTE IS NORMAL.  LUNGS ARE CLEAR.  THERE IS \nNO PLEURAL EFFUSION.  BONY STRUCTURES ARE NORMAL.  LEFT-SIDED VAGAL \nNERVE STIMULATOR IS SEEN.\n \nSUMMARY: 1-NO SIGNIFICANT ABNORMALITY\n \nACCESSION NUMBER:\n759778\nThis report has been anonymized. All dates are offset from the actual dates by a fixed interval associated with the patient.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [7]:

section_terms = [
    "FINDINGS",
    "IMPRESSION",
    "HISTORY",
    "TECHNIQUE",
    "COMPARISON",
]

print("--- Section Marker Occurrences ---")
for term in section_terms:
    count = manifest["report"].str.contains(term, case=False, na=False).sum()
    print(f"{term:12s}: {count:7d}")

--- Section Marker Occurrences ---
FINDINGS    :   97272
IMPRESSION  :  223462
HISTORY     :  178045
TECHNIQUE   :   13461
COMPARISON  :  214253


In [8]:

non_ascii_count = (~manifest["report"].str.match(r"^[\x00-\x7F]*$")).sum()
null_byte_count = manifest["report"].str.contains("\x00", regex=False).sum()

print("Reports containing non-ASCII characters:", non_ascii_count)
print("Reports containing null bytes:          ", null_byte_count)

Reports containing non-ASCII characters: 15
Reports containing null bytes:           0


In [2]:
!pip install -q transformers

from transformers import AutoTokenizer

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Tokenizer loaded successfully.
Vocabulary size: 28996


In [3]:
print("--- Initial 5-Sample Token Length Check ---")
for i, report in enumerate(manifest["report"].head(5)):
    encoded = tokenizer(
        report,
        add_special_tokens=True,
        truncation=False,
    )
    word_count = len(report.split())
    token_count = len(encoded["input_ids"])
    subword_ratio = token_count / word_count if word_count > 0 else 0
    print(
        f"Report {i + 1}: {token_count:3d} tokens | "
        f"{word_count:3d} words | Subword ratio: {subword_ratio:.2f}"
    )

--- Initial 5-Sample Token Length Check ---
Report 1: 228 tokens | 107 words | Subword ratio: 2.13
Report 2: 215 tokens | 108 words | Subword ratio: 1.99
Report 3: 214 tokens | 108 words | Subword ratio: 1.98
Report 4: 189 tokens | 122 words | Subword ratio: 1.55
Report 5: 192 tokens | 122 words | Subword ratio: 1.57


In [4]:
from tqdm.auto import tqdm

reports = manifest["report"].tolist()
token_lengths = []
BATCH_SIZE = 256

for start in tqdm(
    range(0, len(reports), BATCH_SIZE),
    desc="Tokenizing reports",
):
    batch = reports[start : start + BATCH_SIZE]
    encoded = tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False,
    )
    token_lengths.extend(len(ids) for ids in encoded["input_ids"])

print(f"Total tokenized reports: {len(token_lengths):,}")

Tokenizing reports:   0%|          | 0/873 [00:00<?, ?it/s]

Total tokenized reports: 223,462


In [5]:
import numpy as np
import pandas as pd

token_lengths = np.asarray(token_lengths)
token_series = pd.Series(token_lengths)

print("--- Token Length Describe ---")
print(token_series.describe())

percentiles = [0.50, 0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.999, 1.00]
print("\n--- Token Length Quantiles ---")
print(token_series.quantile(percentiles))

--- Token Length Describe ---
count    223462.000000
mean        196.114328
std          63.122547
min          57.000000
25%         156.000000
50%         183.000000
75%         220.000000
max        1395.000000
dtype: float64

--- Token Length Quantiles ---
0.500     183.000
0.750     220.000
0.900     270.000
0.950     314.000
0.975     360.000
0.990     423.000
0.995     477.000
0.999     611.539
1.000    1395.000
dtype: float64


In [6]:
total_reports = len(token_lengths)
candidate_lengths = [128, 160, 192, 256, 320, 384, 512]

print("--- Candidate Sequence Length Truncation Rates ---")
for max_len in candidate_lengths:
    truncated = (token_lengths > max_len).sum()
    percentage = (truncated / total_reports) * 100
    retained_pct = 100.0 - percentage
    print(
        f"max_length={max_len:3d} | "
        f"truncated={truncated:6d} ({percentage:6.2f}%) | "
        f"fully retained={retained_pct:6.2f}%"
    )

--- Candidate Sequence Length Truncation Rates ---
max_length=128 | truncated=207840 ( 93.01%) | fully retained=  6.99%
max_length=160 | truncated=159568 ( 71.41%) | fully retained= 28.59%
max_length=192 | truncated= 94325 ( 42.21%) | fully retained= 57.79%
max_length=256 | truncated= 28247 ( 12.64%) | fully retained= 87.36%
max_length=320 | truncated= 10068 (  4.51%) | fully retained= 95.49%
max_length=384 | truncated=  3896 (  1.74%) | fully retained= 98.26%
max_length=512 | truncated=   730 (  0.33%) | fully retained= 99.67%
